# ML chem drugs

Google Colab notebook version of the attached R Markdown workflow, configured to download the required model and input files from the provided GitHub `model1` URLs.

In [ ]:
needed <- c("readxl", "dplyr", "writexl")

to_install <- needed[!sapply(needed, requireNamespace, quietly = TRUE)]
if (length(to_install) > 0) {
  install.packages(to_install, repos = "https://cloud.r-project.org")
}

library(readxl)
library(dplyr, warn.conflicts = FALSE)
library(writexl)

options(digits = 3)

In [ ]:
urls <- c(
  blind_test_data = "https://github.com/postnicov/MICpredictOnline/raw/refs/heads/main/model1/blind_test_data.xlsx",
  features = "https://github.com/postnicov/MICpredictOnline/raw/refs/heads/main/model1/features.rds",
  model = "https://github.com/postnicov/MICpredictOnline/raw/refs/heads/main/model1/model.rds",
  top_features = "https://github.com/postnicov/MICpredictOnline/raw/refs/heads/main/model1/top_features.rds"
)

dest_files <- c(
  blind_test_data = "blind_test_data.xlsx",
  features = "features.rds",
  model = "model.rds",
  top_features = "top_features.rds"
)

for (nm in names(urls)) {
  download.file(urls[[nm]], destfile = dest_files[[nm]], mode = "wb")
}

file.info(dest_files)[, c("size", "mtime")]

In [ ]:
model <- readRDS("model.rds")
expected_feats <- readRDS("features.rds")
top_features <- readRDS("top_features.rds")

new_data <- read_xlsx("blind_test_data.xlsx", sheet = 1)
ids <- new_data$ID

new_aligned <- new_data %>%
  select(all_of(expected_feats)) %>%
  mutate(across(everything(), as.numeric))

pred_class <- predict(model, newdata = new_aligned, type = "class")

get_fragments <- function(row_data) {
  active_features <- names(row_data)[as.numeric(row_data) == 1]
  active_features <- intersect(active_features, top_features)

  if (length(active_features) == 0) {
    return("None")
  }

  paste(active_features, collapse = "; ")
}

associated_features <- sapply(
  seq_len(nrow(new_aligned)),
  function(i) get_fragments(new_aligned[i, ])
)

result_df <- data.frame(
  ID = ids,
  Predicted_Class = as.character(pred_class),
  Associated_PubChem = associated_features
)

write_xlsx(result_df, "blind_prediction_with_fragments.xlsx")
result_df